In [1]:
from pathlib import Path
from collections import Counter
import hashlib

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Run the notebook with the repository root as the working directory.
PROJECT_DIR = Path.cwd().parent
DATASET_DIR = PROJECT_DIR / "Defect in NEU.v2i.yolov8"
OUTPUT_DIR = Path.cwd() / "outputs" / "assignment6_run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CROP_SIZE = 32
HOG_PARAMETERS = {
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2),
    "block_norm": "L2-Hys",
}
PCA_COMPONENTS = 150
K_VALUES = (1, 3, 5, 7, 9)
SVM_CONFIGS = (
    {"kernel": "linear", "C": 1.0},
    {"kernel": "rbf", "C": 1.0, "gamma": "scale"},
    {"kernel": "rbf", "C": 10.0, "gamma": "scale"},
)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Dataset:", DATASET_DIR)
print("Outputs:", OUTPUT_DIR)

Dataset: C:\Users\user\Downloads\NEUDefect\Defect in NEU.v2i.yolov8
Outputs: C:\Users\user\Downloads\NEUDefect\assignment_6\outputs\assignment6_run


In [2]:
def load_dataset(data_dir):
    """Load YOLO labels and count missing, unreadable, and duplicate data."""
    data_dir = Path(data_dir)
    with (data_dir / "data.yaml").open("r", encoding="utf-8") as file:
        names = yaml.safe_load(file)["names"]
    if isinstance(names, dict):
        names = [names[index] for index in sorted(names)]

    records = []
    audit_rows = []
    seen_hashes = {}
    counts = Counter()

    for split_name in ("train", "valid", "test"):
        images_dir = data_dir / split_name / "images"
        labels_dir = data_dir / split_name / "labels"
        image_paths = sorted(
            path for path in images_dir.iterdir()
            if path.suffix.lower() in IMAGE_SUFFIXES
        )
        image_stems = {path.stem for path in image_paths}
        counts["images_found"] += len(image_paths)
        counts["labels_without_image"] += sum(
            path.stem not in image_stems for path in labels_dir.glob("*.txt")
        )

        for image_path in image_paths:
            label_path = labels_dir / f"{image_path.stem}.txt"
            if not label_path.is_file():
                counts["images_missing_label"] += 1
                audit_rows.append({
                    "image": str(image_path),
                    "source_split": split_name,
                    "status": "missing_label",
                    "sha256": "",
                    "n_boxes": 0,
                    "classes": "",
                })
                continue

            image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if image is None:
                counts["images_unreadable"] += 1
                audit_rows.append({
                    "image": str(image_path),
                    "source_split": split_name,
                    "status": "unreadable_image",
                    "sha256": "",
                    "n_boxes": 0,
                    "classes": "",
                })
                continue

            boxes = []
            with label_path.open("r", encoding="utf-8") as file:
                for line in file:
                    fields = line.split()
                    if fields:
                        class_id = int(fields[0])
                        x_center, y_center, width, height = map(float, fields[1:])
                        boxes.append((class_id, x_center, y_center, width, height))

            if not boxes:
                counts["images_empty_label"] += 1
                audit_rows.append({
                    "image": str(image_path),
                    "source_split": split_name,
                    "status": "empty_label",
                    "sha256": "",
                    "n_boxes": 0,
                    "classes": "",
                })
                continue

            image_hash = hashlib.sha256(image_path.read_bytes()).hexdigest()
            class_ids = sorted({box[0] for box in boxes})
            if image_hash in seen_hashes:
                counts["duplicate_images"] += 1
                audit_rows.append({
                    "image": str(image_path),
                    "source_split": split_name,
                    "status": "exact_duplicate_image",
                    "sha256": image_hash,
                    "n_boxes": len(boxes),
                    "classes": ",".join(names[class_id] for class_id in class_ids),
                })
                continue

            class_counts = Counter(box[0] for box in boxes)
            dominant_class = min(
                class_id for class_id, count in class_counts.items()
                if count == max(class_counts.values())
            )
            record = {
                "image_path": image_path,
                "label_path": label_path,
                "source_split": split_name,
                "image_hash": image_hash,
                "boxes": boxes,
                "class_ids": class_ids,
                "dominant_class": dominant_class,
            }
            records.append(record)
            seen_hashes[image_hash] = record
            counts["images_usable"] += 1
            counts["boxes_usable"] += len(boxes)
            audit_rows.append({
                "image": str(image_path),
                "source_split": split_name,
                "status": "usable",
                "sha256": image_hash,
                "n_boxes": len(boxes),
                "classes": ",".join(names[class_id] for class_id in class_ids),
            })

    return records, names, pd.DataFrame(audit_rows), counts

records, class_names, audit, counts = load_dataset(DATASET_DIR)
audit.to_csv(OUTPUT_DIR / "dataset_audit.csv", index=False)
print(
    f"Images found={counts['images_found']}; usable={counts['images_usable']}; "
    f"missing label files={counts['images_missing_label']}; "
    f"unreadable images={counts['images_unreadable']}; "
    f"empty labels={counts['images_empty_label']}; "
    f"label files without image={counts['labels_without_image']}; "
    f"exact duplicate images skipped={counts['duplicate_images']}"
)
print(f"Usable YOLO bounding boxes: {counts['boxes_usable']}")

Images found=1799; usable=1799; missing label files=0; unreadable images=0; empty labels=0; label files without image=0; exact duplicate images skipped=0
Usable YOLO bounding boxes: 4186


In [3]:
def split_by_image(records):
    indices = np.arange(len(records))
    strata = np.array([record["dominant_class"] for record in records])
    train_indices, test_indices = train_test_split(
        indices, test_size=0.2, random_state=SEED, stratify=strata
    )
    return (
        [records[index] for index in train_indices],
        [records[index] for index in test_indices],
    )

development_records, test_records = split_by_image(records)
manifest = pd.DataFrame([
    {
        "image": str(record["image_path"]),
        "source_split": record["source_split"],
        "assignment_split": split_name,
        "image_hash": record["image_hash"],
        "dominant_class": class_names[record["dominant_class"]],
        "n_boxes": len(record["boxes"]),
        "classes": ",".join(class_names[class_id] for class_id in record["class_ids"]),
    }
    for split_name, split_records in (("development", development_records), ("test", test_records))
    for record in split_records
])
manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
image_counts = pd.crosstab(
    pd.Series([class_names[r["dominant_class"]] for r in development_records + test_records], name="dominant_class"),
    pd.Series(["development"] * len(development_records) + ["test"] * len(test_records), name="split"),
)
print(f"Development/test images: {len(development_records)}/{len(test_records)}")
print("Image-level counts by dominant class (about 300 images per class overall):")
display(image_counts)


Development/test images: 1439/360
Image-level counts by dominant class (about 300 images per class overall):


split,development,test
dominant_class,,
crazing,240,60
inclusion,265,66
patches,257,65
pitted_surface,215,54
rolled-in_scale,240,60
scratches,222,55


In [4]:
def crop_boxes(records):
    """Crop every YOLO box and keep the source image group."""
    crops, labels, groups = [], [], []
    crop_hashes = []
    counts = Counter()
    for record in records:
        image = cv2.imread(str(record["image_path"]), cv2.IMREAD_GRAYSCALE)
        if image is None:
            counts["images_unreadable_during_crop"] += 1
            continue
        image_height, image_width = image.shape
        for class_id, x_center, y_center, width, height in record["boxes"]:
            x1 = max(0, int((x_center - width / 2) * image_width))
            y1 = max(0, int((y_center - height / 2) * image_height))
            x2 = min(image_width, int((x_center + width / 2) * image_width))
            y2 = min(image_height, int((y_center + height / 2) * image_height))
            crop = image[y1:y2, x1:x2]
            if crop.size == 0:
                counts["empty_crops"] += 1
                continue
            crop = cv2.resize(crop, (CROP_SIZE, CROP_SIZE))
            crops.append(crop)
            labels.append(class_id)
            groups.append(record["image_hash"])
            crop_hashes.append(hashlib.sha256(crop.tobytes()).hexdigest())
    counts["crops"] = len(crops)
    counts["duplicate_crops"] = len(crop_hashes) - len(set(crop_hashes))
    return (
        np.asarray(crops, dtype=np.uint8),
        np.asarray(labels, dtype=int),
        np.asarray(groups),
        counts,
    )

development_crops, y_development, development_groups, development_crop_counts = crop_boxes(development_records)
test_crops, y_test, test_groups, test_crop_counts = crop_boxes(test_records)
print(
    f"Crops: development={len(y_development)}, test={len(y_test)}; "
    f"empty crops={development_crop_counts['empty_crops'] + test_crop_counts['empty_crops']}; "
    f"exact duplicate crops retained={development_crop_counts['duplicate_crops'] + test_crop_counts['duplicate_crops']}"
)

Crops: development=3323, test=863; empty crops=0; exact duplicate crops retained=3


In [5]:
def extract_hog(crops):
    return np.asarray(
        [
            hog(crop, **HOG_PARAMETERS)
            for crop in crops
        ],
        dtype=np.float32,
    )

X_development = extract_hog(development_crops)
X_test = extract_hog(test_crops)
print(f"HOG parameters: {HOG_PARAMETERS}; crop={CROP_SIZE}x{CROP_SIZE}; feature length={X_development.shape[1]}")
print("Scaler: StandardScaler, fitted only on undersampled training crops in each fold/final fit.")
class_distribution = pd.DataFrame({
    "class": class_names,
    "development_crops": [int(np.sum(y_development == class_id)) for class_id in range(len(class_names))],
    "test_crops": [int(np.sum(y_test == class_id)) for class_id in range(len(class_names))],
})
class_distribution.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)
display(class_distribution)


HOG parameters: {'orientations': 9, 'pixels_per_cell': (8, 8), 'cells_per_block': (2, 2), 'block_norm': 'L2-Hys'}; crop=32x32; feature length=324
Scaler: StandardScaler, fitted only on undersampled training crops in each fold/final fit.


,class,development_crops,test_crops
0,crazing,543,146
1,inclusion,806,205
2,patches,701,177
3,pitted_surface,341,91
4,rolled-in_scale,498,130
5,scratches,434,114


In [6]:
def undersample_indices(y, seed):
    """Return a seeded, class-balanced subset by undersampling to the minority count."""
    rng = np.random.default_rng(seed)
    classes, class_counts = np.unique(y, return_counts=True)
    target_count = int(class_counts.min())
    selected = np.concatenate([
        rng.choice(np.flatnonzero(y == class_id), size=target_count, replace=False)
        for class_id in classes
    ])
    rng.shuffle(selected)
    return selected

def score_predictions(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

def summarize_cv(fold_results):
    metric_columns = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "balanced_accuracy",
    ]
    summary = fold_results.groupby(
        ["model", "configuration"], as_index=False
    )[metric_columns].agg(["mean", "std"])
    summary.columns = [
        "_".join(str(part) for part in column if part).rstrip("_")
        if isinstance(column, tuple) else column
        for column in summary.columns
    ]
    return summary.sort_values(
        ["model", "f1_macro_mean"], ascending=[True, False]
    ).reset_index(drop=True)

def _fit_scaled_knn(X_train, y_train, k, seed):
    sampled = undersample_indices(y_train, seed)
    scaler = StandardScaler()
    X_balanced = scaler.fit_transform(X_train[sampled])
    model = KNeighborsClassifier(n_neighbors=k, weights="distance")
    model.fit(X_balanced, y_train[sampled])
    return scaler, model, sampled

def _fit_scaled_svm(X_train, y_train, config, seed):
    sampled = undersample_indices(y_train, seed)
    scaler = StandardScaler()
    X_balanced = scaler.fit_transform(X_train[sampled])
    model = SVC(**config)
    model.fit(X_balanced, y_train[sampled])
    return scaler, model, sampled

def cross_validate_hog(X, y, groups):
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows = []
    for fold, (train_idx, validation_idx) in enumerate(
        splitter.split(X, y, groups), start=1
    ):
        for k in K_VALUES:
            scaler, model, sampled = _fit_scaled_knn(
                X[train_idx], y[train_idx], k, SEED + fold
            )
            prediction = model.predict(scaler.transform(X[validation_idx]))
            rows.append({
                "model": "KNN",
                "configuration": f"k={k}",
                "fold": fold,
                "balanced_train_crops": len(sampled),
                **score_predictions(y[validation_idx], prediction),
            })
        for config in SVM_CONFIGS:
            name = ",".join(f"{key}={value}" for key, value in config.items())
            scaler, model, sampled = _fit_scaled_svm(
                X[train_idx], y[train_idx], config, SEED + fold
            )
            prediction = model.predict(scaler.transform(X[validation_idx]))
            rows.append({
                "model": "SVM",
                "configuration": name,
                "fold": fold,
                "balanced_train_crops": len(sampled),
                **score_predictions(y[validation_idx], prediction),
            })
    return pd.DataFrame(rows)

cv_folds = cross_validate_hog(X_development, y_development, development_groups)
cv_folds.to_csv(OUTPUT_DIR / "cv_fold_results.csv", index=False)
cv_summary = summarize_cv(cv_folds)
cv_summary.to_csv(OUTPUT_DIR / "cv_summary.csv", index=False)
print("Training-only random undersampling to minority-class count; test distribution unchanged.")
display(cv_summary)


Training-only random undersampling to minority-class count; test distribution unchanged.


,model,configuration,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,balanced_accuracy_mean,balanced_accuracy_std
0,KNN,k=3,0.489313,0.012883,0.603399,0.006876,0.510470,0.016438,0.498943,0.014355,0.510470,0.016438
1,KNN,k=1,0.482695,0.008712,0.591191,0.007648,0.502706,0.011352,0.496269,0.010568,0.502706,0.011352
2,KNN,k=5,0.481481,0.013045,0.613002,0.003932,0.503060,0.017597,0.489088,0.012671,0.503060,0.017597
3,KNN,k=7,0.480578,0.018460,0.619094,0.010382,0.502041,0.021523,0.484584,0.019590,0.502041,0.021523
4,KNN,k=9,0.478472,0.017708,0.627077,0.007021,0.500321,0.020503,0.480662,0.017675,0.500321,0.020503
5,SVM,"kernel=rbf,C=10.0,gamma=scale",0.757435,0.015126,0.734459,0.015297,0.739837,0.016563,0.734892,0.016247,0.739837,0.016563
6,SVM,"kernel=rbf,C=1.0,gamma=scale",0.759246,0.008832,0.733905,0.011404,0.740201,0.010652,0.733318,0.011114,0.740201,0.010652
7,SVM,"kernel=linear,C=1.0",0.650928,0.012084,0.633072,0.009177,0.627058,0.011327,0.627081,0.009726,0.627058,0.011327


In [7]:
def cross_validate_pca_baseline(crops, y, groups):
    """Development-only PCA baseline using the same folds and training undersampling."""
    flat = crops.reshape(len(crops), -1).astype(np.float32) / 255.0
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows = []
    for fold, (train_idx, validation_idx) in enumerate(
        splitter.split(flat, y, groups), start=1
    ):
        sampled = undersample_indices(y[train_idx], SEED + fold)
        pca = PCA(
            n_components=PCA_COMPONENTS,
            whiten=True,
            svd_solver="randomized",
            random_state=SEED,
        )
        X_train = pca.fit_transform(flat[train_idx][sampled])
        X_validation = pca.transform(flat[validation_idx])
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_validation = scaler.transform(X_validation)
        for model_name, configurations in (
            ("KNN", [{"k": k} for k in K_VALUES]),
            ("SVM", list(SVM_CONFIGS)),
        ):
            for config in configurations:
                if model_name == "KNN":
                    model = KNeighborsClassifier(
                        n_neighbors=config["k"], weights="distance"
                    )
                    name = f"k={config['k']}"
                else:
                    model = SVC(**config)
                    name = ",".join(f"{key}={value}" for key, value in config.items())
                model.fit(X_train, y[train_idx][sampled])
                rows.append({
                    "model": model_name,
                    "configuration": name,
                    "fold": fold,
                    **score_predictions(y[validation_idx], model.predict(X_validation)),
                })
    return pd.DataFrame(rows)

pca_folds = cross_validate_pca_baseline(development_crops, y_development, development_groups)
pca_summary = summarize_cv(pca_folds)
hog_best = cv_summary.sort_values(["model", "f1_macro_mean"], ascending=[True, False]).groupby("model", as_index=False).first()
pca_best = pca_summary.sort_values(["model", "f1_macro_mean"], ascending=[True, False]).groupby("model", as_index=False).first()
feature_comparison = pd.DataFrame([
    {
        "model": model_name,
        "HOG_best_configuration": hog_best.loc[hog_best["model"] == model_name, "configuration"].iloc[0],
        "HOG_best_macro_f1_cv": hog_best.loc[hog_best["model"] == model_name, "f1_macro_mean"].iloc[0],
        "PCA_best_configuration": pca_best.loc[pca_best["model"] == model_name, "configuration"].iloc[0],
        "PCA_best_macro_f1_cv": pca_best.loc[pca_best["model"] == model_name, "f1_macro_mean"].iloc[0],
    }
    for model_name in ("KNN", "SVM")
])
feature_comparison.to_csv(OUTPUT_DIR / "feature_cv_comparison.csv", index=False)
display(feature_comparison)


,model,HOG_best_configuration,HOG_best_macro_f1_cv,PCA_best_configuration,PCA_best_macro_f1_cv
0,KNN,k=3,0.498943,k=1,0.336706
1,SVM,"kernel=rbf,C=10.0,gamma=scale",0.734892,"kernel=rbf,C=10.0,gamma=scale",0.712106


In [8]:
def save_confusion_matrix(y_true, y_pred, class_names, model_name, output_dir):
    matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(
        output_dir / f"confusion_matrix_{model_name.lower()}.csv"
    )
    figure, axis = plt.subplots(figsize=(8, 7))
    image = axis.imshow(matrix, cmap="Blues")
    figure.colorbar(image, ax=axis)
    axis.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted",
        ylabel="Actual",
        title=f"Confusion Matrix - {model_name}",
    )
    plt.setp(axis.get_xticklabels(), rotation=35, ha="right", rotation_mode="anchor")
    threshold = matrix.max() / 2
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            axis.text(
                column, row, str(matrix[row, column]),
                ha="center", va="center",
                color="white" if matrix[row, column] > threshold else "black",
            )
    figure.tight_layout()
    figure.savefig(output_dir / f"confusion_matrix_{model_name.lower()}.png", dpi=160)
    plt.close(figure)
    return matrix

knn_best = cv_summary[cv_summary["model"] == "KNN"].sort_values(
    ["f1_macro_mean", "accuracy_mean"], ascending=False
).iloc[0]
best_k = int(knn_best["configuration"].split("=")[1])
svm_best = cv_summary[cv_summary["model"] == "SVM"].sort_values(
    ["f1_macro_mean", "accuracy_mean"], ascending=False
).iloc[0]
best_svm_config = next(
    config for config in SVM_CONFIGS
    if ",".join(f"{key}={value}" for key, value in config.items()) == svm_best["configuration"]
)
print(f"Selected KNN: k={best_k}; selected SVM: {svm_best['configuration']}")

final_models = []
knn_scaler, knn_model, knn_sampled = _fit_scaled_knn(X_development, y_development, best_k, SEED)
svm_scaler, svm_model, svm_sampled = _fit_scaled_svm(X_development, y_development, best_svm_config, SEED)
print(f"Balanced final training count per class: {len(knn_sampled) // len(class_names)}")
final_models.append(("KNN", f"k={best_k}", knn_scaler, knn_model))
final_models.append(("SVM", svm_best["configuration"], svm_scaler, svm_model))

test_rows = []
confusion_matrices = {}
for model_name, configuration, scaler, model in final_models:
    prediction = model.predict(scaler.transform(X_test))
    scores = score_predictions(y_test, prediction)
    test_rows.append({"model": model_name, "configuration": configuration, **scores})
    report = classification_report(
        y_test, prediction, labels=np.arange(len(class_names)),
        target_names=class_names, output_dict=True, zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(
        OUTPUT_DIR / f"classification_report_{model_name.lower()}.csv"
    )
    print(f"\n{model_name} per-class classification report (test):")
    print(classification_report(
        y_test, prediction, labels=np.arange(len(class_names)),
        target_names=class_names, zero_division=0,
    ))
    confusion_matrices[model_name] = save_confusion_matrix(
        y_test, prediction, class_names, model_name, OUTPUT_DIR
    )
    print(f"{model_name} confusion matrix (actual rows, predicted columns):")
    print(pd.DataFrame(confusion_matrices[model_name], index=class_names, columns=class_names).to_string())

test_metrics = pd.DataFrame(test_rows)
test_metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
print("Final test metrics; test set was used once after model selection:")
display(test_metrics)

summary_lines = [
    "Assignment 6 - HOG pipeline run summary",
    f"Dataset: {DATASET_DIR.resolve()}",
    f"Seed: {SEED}",
    f"Dataset images found/usable: {counts['images_found']}/{counts['images_usable']}",
    f"Missing label files: {counts['images_missing_label']}",
    f"Unreadable images: {counts['images_unreadable']}",
    f"Images with empty labels: {counts['images_empty_label']}",
    f"Label files without images: {counts['labels_without_image']}",
    f"Exact duplicate images skipped: {counts['duplicate_images']}",
    f"Usable YOLO boxes: {counts['boxes_usable']}",
    f"Empty crops: {development_crop_counts['empty_crops'] + test_crop_counts['empty_crops']}",
    f"Duplicate crops retained: {development_crop_counts['duplicate_crops'] + test_crop_counts['duplicate_crops']}",
    f"Crop size: {CROP_SIZE}x{CROP_SIZE}; HOG: {HOG_PARAMETERS}",
    "Scaler: StandardScaler fit on undersampled training data only",
    "Balance: random undersampling to minority count in each training fold and final development fit",
    "Test distribution: unchanged (imbalanced)",
    f"Development/test images: {len(development_records)}/{len(test_records)}",
    f"Development/test crops: {len(y_development)}/{len(y_test)}",
    f"Best KNN by development macro-F1: {best_k}",
    f"Best SVM by development macro-F1: {svm_best['configuration']}",
    "",
    "Image-level counts by dominant class:",
    "\n".join(line.rstrip() for line in image_counts.to_string().splitlines()),
    "",
    "Crop distribution:",
    class_distribution.to_string(index=False),
    "",
    "Development-only HOG/PCA CV comparison:",
    feature_comparison.to_string(index=False),
    "",
    "Development CV summary:",
    cv_summary.to_string(index=False),
    "",
    "Final test metrics:",
    test_metrics.to_string(index=False),
]
(OUTPUT_DIR / "run_summary.txt").write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print("Run summary saved:", OUTPUT_DIR / "run_summary.txt")


Selected KNN: k=3; selected SVM: kernel=rbf,C=10.0,gamma=scale


Balanced final training count per class: 341

KNN per-class classification report (test):
                 precision    recall  f1-score   support

        crazing       0.33      0.64      0.44       146
      inclusion       0.85      0.31      0.46       205
        patches       0.93      0.46      0.61       177
 pitted_surface       0.19      0.31      0.24        91
rolled-in_scale       0.39      0.56      0.46       130
      scratches       0.91      0.73      0.81       114

       accuracy                           0.49       863
      macro avg       0.60      0.50      0.50       863
   weighted avg       0.65      0.49      0.51       863



KNN confusion matrix (actual rows, predicted columns):
                 crazing  inclusion  patches  pitted_surface  rolled-in_scale  scratches
crazing               93          0        0              38               15          0
inclusion             56         64        4              31               43          7
patches               49          1       81              20               25          1
pitted_surface        39          0        0              28               24          0
rolled-in_scale       36          0        2              19               73          0
scratches              5         10        0               8                8         83



SVM per-class classification report (test):
                 precision    recall  f1-score   support

        crazing       0.62      0.74      0.68       146
      inclusion       0.88      0.77      0.82       205
        patches       0.91      0.85      0.88       177
 pitted_surface       0.50      0.45      0.47        91
rolled-in_scale       0.63      0.68      0.65       130
      scratches       0.85      0.92      0.89       114

       accuracy                           0.75       863
      macro avg       0.73      0.73      0.73       863
   weighted avg       0.76      0.75      0.75       863



SVM confusion matrix (actual rows, predicted columns):
                 crazing  inclusion  patches  pitted_surface  rolled-in_scale  scratches
crazing              108          2        1              20               15          0
inclusion              3        158        7               9               13         15
patches                7         11      150               4                4          1
pitted_surface        26          2        1              41               19          2
rolled-in_scale       30          1        3               8               88          0
scratches              0          6        3               0                0        105
Final test metrics; test set was used once after model selection:


,model,configuration,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
0,KNN,k=3,0.488992,0.500685,0.602288,0.500685,0.502772
1,SVM,"kernel=rbf,C=10.0,gamma=scale",0.753187,0.734407,0.732385,0.734407,0.731219


Run summary saved: C:\Users\user\Downloads\NEUDefect\assignment_6\outputs\assignment6_run\run_summary.txt
